# 电商平台用户消费行为诊断分析

**基于巴西 Olist 电商平台 9 张业务表、160 万行真实交易数据**

| 诊断问题 | 核心发现 |
|---|---|
| 一、GMV 增长来自哪里？ | 纯拉新驱动，0%复购，SP州+health_beauty集中 |
| 二、用户质量怎么样？ | 100%一次性用户，Top20%贡献56%GMV |
| 三、什么影响用户体验？ | 延迟差评率54% vs 准时9%，剂量反应显著 |

技术栈：MySQL · Python (pandas, matplotlib) · Excel

## 1. 数据概览

数据覆盖 2016-09 至 2018-08，共 9 张业务表，涵盖订单、客户、支付、评价、商品五个业务域。

In [ ]:
import pymysql, pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib, sys, warnings
sys.stdout.reconfigure(encoding='utf-8')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

conn = pymysql.connect(
    host='127.0.0.1', user='root', password='123456',
    database='ecommerce_analysis', charset='utf8mb4'
)

# 各表数据量
df_stats = pd.read_sql_query('''
    SELECT 'customers' AS tbl, COUNT(*) AS rows_ FROM customers
    UNION ALL SELECT 'orders', COUNT(*) FROM orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
    UNION ALL SELECT 'order_payments', COUNT(*) FROM order_payments
    UNION ALL SELECT 'order_reviews', COUNT(*) FROM order_reviews
    UNION ALL SELECT 'products', COUNT(*) FROM products
    UNION ALL SELECT 'sellers', COUNT(*) FROM sellers
    UNION ALL SELECT 'geolocation', COUNT(*) FROM geolocation
    UNION ALL SELECT 'category_transl', COUNT(*) FROM product_category_translation
''', conn)
df_stats.columns = ['Table', 'Rows']
df_stats['Rows'] = df_stats['Rows'].apply(lambda x: f'{x:,}')
df_stats

**数据质量速查：**
- 订单状态：delivered 占 97.4%，后续分析仅取已完成订单
- 时间异常：无签收时间早于下单时间的异常记录
- 评分缺失率：0%
- 地理位置表（100万行）为 zip code 经纬度映射，未参与核心分析

---

## 2. 诊断一：GMV 增长来自哪里？

### 2.1 GMV 量价拆解

将月度 GMV 拆解为订单量和客单价两个因子。

In [ ]:
df_trend = pd.read_sql_query('''
    SELECT DATE_FORMAT(下单时间, '%Y-%m') AS month,
           COUNT(DISTINCT o.订单ID) AS orders,
           ROUND(SUM(oi.价格), 2) AS gmv,
           ROUND(SUM(oi.价格)/COUNT(DISTINCT o.订单ID), 2) AS aov
    FROM orders o JOIN order_items oi ON o.订单ID = oi.订单ID
    WHERE o.订单状态 = 'delivered'
    GROUP BY month ORDER BY month
''', conn)

df_valid = df_trend.iloc[1:-1]
print(f'月均GMV: {df_valid["gmv"].mean():,.0f} BRL')
print(f'月均订单量: {df_valid["orders"].mean():,.0f}')
print(f'月均客单价: {df_valid["aov"].mean():.0f} BRL')
print(f'订单量月均环比: {df_valid["orders"].pct_change().mean()*100:.1f}%')
print(f'客单价月均环比: {df_valid["aov"].pct_change().mean()*100:.1f}%')

![gmv_decomposition](outputs/growth_01_gmv_decomposition.png)

**发现：** 客单价稳定在 ~133 BRL 附近，GMV 的月度波动几乎完全由订单量变化解释。平台增长模式是「量驱动」而非「价驱动」。

### 2.2 复购诊断

量驱动的增长，是靠老客复购还是持续拉新？

In [ ]:
df_freq = pd.read_sql_query('''
    SELECT CASE WHEN order_count = 1 THEN '1 order' ELSE '2+ orders' END AS freq,
           COUNT(*) AS users
    FROM (SELECT 客户ID, COUNT(DISTINCT 订单ID) AS order_count
          FROM orders WHERE 订单状态='delivered' GROUP BY 客户ID) t
    GROUP BY freq
''', conn)
df_freq

![zero_repurchase](outputs/growth_02_zero_repurchase.png)

**[核心发现] 平台 delivered 订单的复购率为绝对零。** 96,478 个完成交易的客户中，没有任何人下过第二单。这不是「复购率低」，而是根本没有复购机制。平台本质上是「获客 -> 转化 -> 流失」的漏斗，而非用户运营平台。

### 2.3 品类增长贡献

![category_attribution](outputs/growth_03_category_attribution.png)

**发现：** health_beauty（健康美容）是第一大品类和最大增长贡献者，watches_gifts、bed_bath_table 紧随其后。长尾品类数量多但贡献小，符合电商典型的头部集中分布。

### 2.4 地区集中度

![regional](outputs/growth_04_regional_concentration.png)

**发现：** SP（圣保罗州）独占 38.3% GMV，Top 5 州合计 73.9%。增长高度依赖单一市场——这是效率也是风险。一旦 SP 州市场饱和或竞争加剧，整体 GMV 面临断崖式下滑风险。

### 诊断一结论

> **GMV 增长完全由新客订单量驱动，客单价无提升，复购率为零，品类和地区高度集中。** 平台处于「流量换增长」的初级阶段，缺乏用户留存和品类/区域多元化两道防线。

---

## 3. 诊断二：平台用户质量怎么样？

### 3.1 购买频次与 RFM 分层

![frequency](outputs/user_01_frequency.png)

100% 的用户仅购买 1 次，不存在「高频用户」这个概念。

![rfm](outputs/user_03_rfm.png)

**RFM 分层结果：** High-Value 33.7%，Potential 32.6%，Average 22.9%，At-Risk 10.8%。

**数据局限警示：** F值（购买频次）因复购率为零而完全失去区分度——所有人 F=1。RFM 在此数据集中退化为 R+M 二维模型。这不是方法论的问题，而是数据本身的局限性。在复购正常的平台（天猫、京东），F值是区分用户价值最关键的维度。面试中能主动指出这一点，比仅仅跑通 RFM 更能体现分析素养。

### 3.2 用户贡献集中度

![concentration](outputs/user_04_concentration.png)

**帕累托分布：** Top 20% 用户贡献 56.4% GMV，Bottom 50% 用户仅贡献 16.8%。

### 3.3 高价值用户画像

![hv_profile](outputs/user_05_hv_profile.png)

**高价值用户特征（vs 其他用户）：**
- 品类：更偏爱 watches_gifts（+2.0pp）和 health_beauty（+1.5pp）
- 支付：信用卡平均支付金额 252 BRL vs 其他用户 157 BRL
- 地区：AL、PB、PA 等东北部州渗透率更高

高价值用户并非均匀分布——特定地区用户天然更倾向高消费。

### 诊断二结论

> **平台用户结构极度脆弱：100% 一次性用户，无复购行为，RFM 模型因数据局限退化为 R+M。** 高价值用户集中在特定品类和地区。一旦这些用户的获取成本上升，GMV 将直接承压。

---

## 4. 诊断三：什么因素影响用户体验？

### 4.1 物流全链路时效

![logistics](outputs/ux_01_logistics_timeline.png)

平均全程耗时 12.1 天：确认 9.9 小时 -> 发货 2.3 天 -> 运输 8.9 天。整体延迟率 8.1%。

### 4.2 延迟 -> 评分：核心归因

In [ ]:
df_delay = pd.read_sql_query('''
    SELECT CASE WHEN o.客户签收时间 > o.预计送达时间 THEN 'Delayed' ELSE 'On Time' END AS status,
           COUNT(DISTINCT o.订单ID) AS orders,
           ROUND(AVG(rv.评分), 2) AS avg_score,
           ROUND(SUM(CASE WHEN rv.评分<=2 THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS bad_rate
    FROM orders o JOIN order_reviews rv ON o.订单ID = rv.订单ID
    WHERE o.订单状态='delivered' AND o.客户签收时间 IS NOT NULL
    GROUP BY status
''', conn)
df_delay

![delay_vs_rating](outputs/ux_02_delay_vs_rating.png)

**这是本项目最重要的归因发现：**
- 准时订单：平均评分 **4.29**，差评率 **9.2%**
- 延迟订单：平均评分 **2.57**，差评率 **54.0%**
- **延迟让评分下降 1.72 分，差评率飙升 6 倍**

### 4.3 延迟归因：地区 x 品类

![delay_region](outputs/ux_03_delay_by_region.png)

- 延迟率最高：AL（23.4%）、MA（19.1%）、PI（15.9%）
- 延迟率最低：SP（5.8%）、PR（4.9%）、RO（2.9%）
- 地区间差距可达 **8 倍**，东北部偏远州显著高于南部发达州

![delay_category](outputs/ux_04_delay_by_category.png)

- 延迟率最高：audio（12.5%）、books_technical（11.0%）、home_confort（10.2%）
- office_furniture 延迟率 8.8% 但评分最低（3.52）——延迟 + 大件商品不满叠加

### 4.4 剂量反应：延迟越久，评分越低

![dose_response](outputs/ux_05_dose_response.png)

| 延迟程度 | 平均评分 | 差评率 |
|---|---|---|
| 准时/提前 | 4.29 | 9.3% |
| 延迟 1-3 天 | 3.29 | 32.2% |
| 延迟 4-7 天 | 2.10 | 67.7% |
| 延迟 8-14 天 | 1.68 | 80.0% |
| 延迟 >14 天 | 1.72 | 78.3% |

**延迟仅 1-3 天评分即从 4.29 跌至 3.29——用户对延迟的容忍度极低。** 延迟超过一周后，差评率稳定在 70-80%，用户体验已无可挽回。

### 诊断三结论

> **物流延迟是影响用户评分的首要因素。延迟订单差评率是准时订单的 6 倍，延迟仅 1-3 天评分即显著下降。** 延迟问题集中在东北部偏远州和大件/专业品类。

---

## 5. 综合建议与业务行动项

### 三个诊断的交叉验证

将三个诊断放在一起看，会发现它们指向同一个核心矛盾：

```
诊断一：GMV 靠新客拉新，没复购
    +
诊断二：100% 一次性用户，没留存
    +
诊断三：延迟让 8.1% 的用户给出 54% 的差评率
    =
一个无复购的漏斗中，每一次差评都是永久性客户流失
```

在正常平台，差评用户还有挽回机会（复购->改评->CLV恢复）。在 Olist，差评 = 永别。

### 优先级排序的行动建议

| 优先级 | 行动 | 对应诊断 | 预期效果 |
|---|---|---|---|
| P0 | 建立首单后复购触达机制（7/30/90天） | 一、二 | 复购从0%到3%，GMV直接+3% |
| P0 | 延迟>3天主动触达用户（道歉+优惠券） | 三 | 截断差评->流失链路 |
| P1 | AL/MA 高延迟州设前置仓或调整预计时间 | 三 | 降低整体延迟率 |
| P1 | SP州深耕 + 腰部州（RS/PR/SC）试点 | 一 | 降低单一市场依赖 |
| P2 | health_beauty + watches_gifts 品类订阅制 | 一、二 | 创造自然复购场景 |

---

## 附录：完整分析脚本

本 Notebook 展示关键 SQL 和分析结论。完整可执行脚本：

- [growth_diagnosis.py](growth_diagnosis.py) -- 诊断一完整代码（4张图）
- [user_quality.py](user_quality.py) -- 诊断二完整代码（5张图）
- [ux_attribution.py](ux_attribution.py) -- 诊断三完整代码（5张图）
- [setup_database.py](setup_database.py) -- 数据入库脚本

In [ ]:
conn.close()